In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact, mannwhitneyu
import gseapy as gp
import os
import urllib.request

import scanpy as sc
import squidpy as sq
import anndata as ad
import re

In [ ]:
GSEA_CSV = "gsea_results/gsea_pfi_overall_MSigDB_Hallmark_2020.csv"
HALLMARK = "MSigDB_Hallmark_2020"

LIB_NAME = "MSigDB_Hallmark_2020"
LIB_CACHE = "MSigDB_Hallmark_2020.tsv"
IFN_SETS = ["Interferon Gamma Response", "Interferon Alpha Response"]
CONTROL_SET = "Allograft Rejection"
RANKING = "gsea_results/gene_ranking_overall.csv"

# curated partitions of the antigen-presentation module
MHC2 = ["CD74", "CIITA", "HLA-DMA", "HLA-DMB", "HLA-DOA", "HLA-DOB",
        "HLA-DPA1", "HLA-DPB1", "HLA-DQA1", "HLA-DQA2", "HLA-DQB1",
        "HLA-DRA", "HLA-DRB1", "HLA-DRB5"]
MHC1 = ["HLA-A", "HLA-B", "HLA-C", "HLA-E", "HLA-F", "HLA-G", "B2M",
        "TAP1", "TAP2", "TAPBP", "PSMB8", "PSMB9", "NLRC5"]

AP = set(MHC1) | set(MHC2)
CLASS_OF = {**{g: "MHC-II" for g in MHC2}, **{g: "MHC-I / peptide loading" for g in MHC1}}

In [ ]:
adata = ad.read_h5ad("../quality_control/primary-cohort/adata.h5ad")
adata.obs['class'] = adata.obs['class'].astype(str)
adata.obs['class_overall'] = adata.obs['class'].replace({'OvaryR': 'Ovary', 'OvaryL': 'Ovary'})
adata.obs['PFI'] = adata.obs.PFI.astype(str)
adata.obs['PFI_short_long'] = adata.obs['PFI'].replace({'short': 'short', 'medium': 'long', 'long': 'long'})
adata.obs['patient'] = adata.obs['patient'].astype(str)
adata.obs['anno'] = adata.obs['class'] + '_' + adata.obs['patient'] + '_PFI-' + adata.obs['PFI']
adata = adata[~adata.obs['class'].isin(['Marker'])].copy()

In [ ]:
print(len(MHC2))
print(len(MHC1))
print(len(AP))

In [ ]:
mhc2_genes_detected = list(adata.var.index[adata.var.index.isin(MHC2)].copy())
print(f"{len(mhc2_genes_detected)} MHC II genes in the probeset, namly")
print(mhc2_genes_detected)

In [ ]:
mhc1_genes_detected = list(adata.var.index[adata.var.index.isin(MHC1)].copy())
print(f"{len(mhc1_genes_detected)} MHC I genes in the probeset, namely")
print(mhc1_genes_detected)

In [ ]:
confirmed_detected = mhc2_genes_detected
confirmed_detected.extend(mhc1_genes_detected)
confirmed_detected = set(confirmed_detected)
print(confirmed_detected)

In [ ]:
UNRECOVERED = set(MHC1).difference(mhc1_genes_detected)
UNRECOVERED

In [ ]:
def load_library(name=LIB_NAME, cache=LIB_CACHE):
    """Hallmark gene sets, from local cache or Enrichr."""
    if not os.path.exists(cache):
        url = ("https://maayanlab.cloud/Enrichr/geneSetLibrary"
               f"?mode=text&libraryName={name}")
        with urllib.request.urlopen(url, timeout=60) as fh:
            open(cache, "wb").write(fh.read())
    lib = {}
    for line in open(cache):
        parts = line.rstrip("\n").split("\t")
        if len(parts) > 2:
            lib[parts[0]] = [g for g in parts[2:] if g]
    return lib

def load_gsea(path=GSEA_CSV):
    """GSEA table with leading-edge sets and the Tag % numerator/denominator split."""
    df = pd.read_csv(path)
    lead_col = next(c for c in df.columns
                    if c.lower() in ("lead_genes", "leading_edge", "ledge_genes"))
    df["lead_set"] = df[lead_col].fillna("").map(
        lambda s: {x.strip() for x in str(s).split(";") if x.strip()})
    tag = df["Tag %"].astype(str).str.extract(r"(\d+)\s*/\s*(\d+)")
    df["n_lead"] = tag[0].astype(int)
    df["n_detected"] = tag[1].astype(int)
    assert (df.n_lead == df.lead_set.map(len)).all(), \
        "Tag % numerator disagrees with the leading-edge gene list"
    return df

def confirm_detection(df):
    """Genes provably present in the ranking.
    """
    return set().union(*df.lead_set)


# leading-edge composition
def composition(df, lib, module=None):
    """Leading-edge composition by functional class. `module` overrides AP."""
    ap = set(module) if module is not None else AP
    rows = []
    for term in IFN_SETS:
        r = df.loc[df.Term == term].iloc[0]
        members = set(lib[term])
        for label, genes in [("Antigen presentation", members & ap),
                             ("  MHC-II", members & set(MHC2)),
                             ("  MHC-I / peptide loading", members & set(MHC1)),
                             ("Effector ISGs", members - ap)]:
            in_lead = genes & r.lead_set
            rows.append({
                "gene_set": term, "NES": round(r.NES, 3), "FDR_q": r["FDR q-val"],
                "module": label.strip(), "indent": label.startswith("  "),
                "n_members": len(genes), "n_in_leading_edge": len(in_lead),
                "frac_in_leading_edge_pct": (round(100 * len(in_lead) / len(genes), 1)
                                             if genes else np.nan),
                "genes_in_leading_edge": ";".join(sorted(in_lead)) or "-",
                "genes_absent": ";".join(sorted(genes - r.lead_set)) or "-",
            })
    return pd.DataFrame(rows)

# depletion test
def depletion_test(df, lib, ranked, module=None, sets=None):
    """One-sided Fisher test for depletion of a set of genes as 'module' from the leading edge.

                        in leading edge   not in leading edge
        module                 a                 k - a
        other members          c            (n_det - k) - c
    """
    ap = set(module) if module is not None else AP
    sets = sets if sets is not None else IFN_SETS
    rows = []
    for term in sets:
        r = df.loc[df.Term == term].iloc[0]
        members = set(lib[term]) & ranked
        assert len(members) == int(r.n_detected), (
            f"{term}: {len(members)} set members in the ranking but gseapy reported "
            f"n_detected={int(r.n_detected)}.")

        mod = members & ap
        k = len(mod)
        a = len(mod & r.lead_set)
        c = int(r.n_lead) - a
        odds, p = fisher_exact([[a, k - a], [c, (len(members) - k) - c]],
                               alternative="less")
        rows.append({
            "gene_set": term,
            "n_detected": len(members),
            "k_module_detected": k,
            "module_in_lead": a,
            "module_frac_pct": round(100 * a / k, 1),
            "other_detected": len(members) - k,
            "other_in_lead": c,
            "other_frac_pct": round(100 * c / (len(members) - k), 1),
            "odds_ratio": round(float(odds), 4) if np.isfinite(odds) else np.inf,
            "p_one_sided_depletion": round(float(p), 5),
            "module_not_ranked": ";".join(sorted(ap & (set(lib[term]) - ranked))) or "-",
        })
    return pd.DataFrame(rows)


def rank_test(df, lib, ranks, module=None, sets=None):
    """Mann-Whitney U on the ranking metric: module members vs other set members.
    """
    ap = set(module) if module is not None else AP
    sets = sets if sets is not None else IFN_SETS
    rows = []
    for term in sets:
        members = set(lib[term]) & set(ranks.index)
        mod = sorted(members & ap)
        oth = sorted(members - set(mod))
        x, y = ranks.loc[mod].to_numpy(), ranks.loc[oth].to_numpy()
        u, p = mannwhitneyu(x, y, alternative="less")
        rows.append({
            "gene_set": term,
            "n_module": len(mod),
            "n_other": len(oth),
            "median_metric_module": round(float(np.median(x)), 4),
            "median_metric_other": round(float(np.median(y)), 4),
            # rank-biserial: 0 = no separation, -1 = module entirely below other
            "rank_biserial_r": round(2 * u / (len(x) * len(y)) - 1, 4),
            "p_one_sided_lower": round(float(p), 5),
        })
    return pd.DataFrame(rows)


# set-level control
def set_level(df, lib, sets=None):
    sets = sets if sets is not None else IFN_SETS + [CONTROL_SET]
    rows = []
    for term in sets:
        r = df.loc[df.Term == term].iloc[0]
        m2 = set(lib[term]) & set(MHC2)
        m1 = set(lib[term]) & set(MHC1)
        rows.append({
            "gene_set": term, "NES": round(r.NES, 3),
            "nominal_p": r["NOM p-val"], "FDR_q": r["FDR q-val"],
            "significant_q_lt_0.25": bool(r["FDR q-val"] < 0.25),
            "n_MHC2_members": len(m2), "n_MHC2_in_leading_edge": len(m2 & r.lead_set),
            "n_MHC1_members": len(m1), "n_MHC1_in_leading_edge": len(m1 & r.lead_set),
            "MHC2_in_leading_edge": ";".join(sorted(m2 & r.lead_set)) or "-",
            "MHC1_in_leading_edge": ";".join(sorted(m1 & r.lead_set)) or "-",
        })
    return pd.DataFrame(rows).sort_values("n_MHC2_members", ascending=False)


def mhc2_ranking(df, lib):
    """Every Hallmark set containing MHC class II genes, ranked by class II content."""
    rows = []
    for term in df.Term:
        members = set(lib.get(term, []))
        m2 = members & set(MHC2)
        if not m2:
            continue
        r = df.loc[df.Term == term].iloc[0]
        rows.append({"gene_set": term, "n_MHC2_members": len(m2),
                     "MHC2_members": ";".join(sorted(m2)),
                     "n_MHC2_in_leading_edge": len(m2 & r.lead_set),
                     "NES": round(r.NES, 3), "FDR_q": r["FDR q-val"],
                     "significant_q_lt_0.25": bool(r["FDR q-val"] < 0.25)})
    return pd.DataFrame(rows).sort_values("n_MHC2_members", ascending=False)


def per_gene_table(df, lib, sets=None):
    sets = sets if sets is not None else IFN_SETS + [CONTROL_SET]
    rows = []
    for term in sets:
        r = df.loc[df.Term == term].iloc[0]
        for gene in sorted(set(lib[term]) & (AP | set(UNRECOVERED))):
            rows.append({
                "gene_set": term, "gene": gene, "class": CLASS_OF[gene],
                "in_measured_probeset": gene not in UNRECOVERED,
                "in_leading_edge": "yes" if gene in r.lead_set else "no",
                "set_NES": round(r.NES, 3), "set_FDR_q": r["FDR q-val"],
                "set_leading_edge_size": r.n_lead,
                "set_members_detected": r.n_detected,
            })
    return pd.DataFrame(rows)

In [ ]:
lib = load_library()
df = load_gsea()
confirmed = confirm_detection(df)

print(f"curated module: {len(MHC2)} MHC-II + {len(MHC1)} MHC-I = {len(AP)} genes")

In [ ]:
MEASURED = set(adata.var.index)

MHC2 = [x for x in MHC2 if x in MEASURED]
MHC1 = [x for x in MHC1 if x in MEASURED]
UNRECOVERED = sorted((set(MHC1) | set(MHC2)) - MEASURED)
AP = set(MHC1) | set(MHC2)

In [ ]:
CLASS_OF = {**{g: "MHC-II" for g in MHC2},
            **{g: "MHC-I / peptide loading" for g in MHC1}}

In [ ]:
lib = load_library()
df = load_gsea()

# Lower bound
lead_bound = set().union(*df.lead_set)
assert lead_bound & AP <= MEASURED, \
    f"in a leading edge but not in adata.var: {sorted((lead_bound & AP) - MEASURED)}"

ranked_genes = pd.read_csv(RANKING, names=["gene", "rank"], header=0)
RANKED = set(ranked_genes['gene'])

confirmed = AP & RANKED
print(f"module genes present in the ranking: {len(confirmed)}")

In [ ]:
t1 = composition(df, lib)
print("leading-edge composition")
print(t1[["gene_set", "module", "n_members", "n_in_leading_edge",
          "frac_in_leading_edge_pct"]].to_string(index=False))
print("    (leading-edge gene lists in suppl_T1_leading_edge_composition.csv)")
print("-" * 70)

t2 = depletion_test(df, lib, RANKED)
print("\ndepletion test")
print(t2.to_string(index=False))
print("-" * 70)

ranks  = ranked_genes.set_index("gene")["rank"]
t_sens = rank_test(df, lib, ranks)
t_sens.to_csv("rank_sensitivity.csv", index=False)
print("\nset-level control")
print(t_sens.to_string(index=False))
print("-"*70)

rank = mhc2_ranking(df, lib)
print("\nHallmark sets ranked by MHC class II content")
print(rank[["gene_set", "n_MHC2_members", "n_MHC2_in_leading_edge",
            "NES", "FDR_q"]].to_string(index=False))

pg = per_gene_table(df, lib)
for name, obj in [("leading_edge_composition", t1),
                  ("depletion_test", t2),
                  ("rank_sensitivity", t_sens),
                  ("mhc2_ranking", rank),
                  ("table_AP_leading_edge", pg)]:
    obj.to_csv(f"MHC_results/{name}.csv", index=False)
print("\nwrote 7 CSVs")

In [ ]:
print(f"RANKED: {len(RANKED)} genes | duplicates in file: "
      f"{len(ranked_genes) - ranked_genes.gene.nunique()}")
print(f"header leaked in: {'gene' in RANKED or 'Gene' in RANKED}")
print(f"first 3 rows read: {ranked_genes.head(3).to_dict('records')}\n")

# every leading-edge gene MUST be in the ranking
lead_all = set().union(*df.lead_set)
missing_from_ranked = sorted(lead_all - RANKED)
assert not missing_from_ranked, (
    f"{len(missing_from_ranked)} leading-edge genes absent from RANKED -> wrong file "
    f"or wrong parse: {missing_from_ranked[:10]}")
print(f"invariant OK: all {len(lead_all)} leading-edge genes are in RANKED\n")

for term in ["Interferon Gamma Response", "Interferon Alpha Response"]:
    n_reported = int(df.loc[df.Term == term, "n_detected"].iloc[0])
    n_ranked   = len(set(lib[term]) & RANKED)
    mod        = set(lib[term]) & AP
    mod_ranked = mod & RANKED
    ok = n_ranked == n_reported
    print(f"{term}")
    print(f"   gseapy detected      : {n_reported}")
    print(f"   set members in RANKED: {n_ranked}   {'MATCH' if ok else 'MISMATCH'}")
    print(f"   -> k = {len(mod_ranked)} of {len(mod)} module members"
          f" {'(PROVEN)' if ok else '(something is wrong)'}")
    dropped = sorted(mod - RANKED)
    if dropped:
        print(f"   module genes NOT ranked: {dropped}")
    print()